# 사극체 말투 변환기 (LoRA, 양자화 없음)

이 노트북은 **Qwen/Qwen3-1.7B** 모델을 양자화하지 않고 `bf16`으로 불러온 뒤, **LoRA 어댑터만 학습**해서 답변 말투를 사극체로 바꾸는 실습입니다.

핵심 흐름은 다음과 같습니다.

1. 파인튜닝 전 기본 모델의 답변을 확인합니다.
2. 현대어 질문 → 사극체 답변 쌍 40개를 `prompt` / `completion` 형태로 만듭니다.
3. 원본 모델 가중치는 고정하고 LoRA 어댑터만 학습합니다.
4. 같은 질문에 대해 파인튜닝 전/후 답변을 비교합니다.
5. 학습된 LoRA 어댑터만 파일로 저장합니다.

> 이 노트북은 **QLoRA가 아닙니다.** 모델 가중치를 4bit로 불러오지 않습니다. 다만 학습 시 VRAM을 아끼기 위해 optimizer는 `paged_adamw_8bit`를 사용합니다.


In [1]:
import os
import warnings
import logging

# 1. 파이썬 기본 경고 무시
warnings.filterwarnings("ignore")

# 2. 시스템 환경변수를 통한 Hugging Face 및 커널 로그 제어 (0=ALL, 1=INFO, 2=WARNING, 3=ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["LOGGERS_LEVEL"] = "ERROR"

# 3. transformers 자체 라이브러리 로그 레벨을 ERROR로 설정
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

## 0. 환경 확인

먼저 Python, PyTorch, CUDA, GPU, VRAM 상태를 확인합니다. 이 실습은 **Python 3.11 + JupyterLab** 환경을 기준으로 작성되었습니다.

- NVIDIA GPU가 정상적으로 인식되어야 합니다.
- 기본 설정은 6GB VRAM급 GPU에서도 실행되도록 보수적으로 잡았습니다.
- VRAM이 부족하면 뒤쪽 학습 설정에서 `max_length`를 줄이거나 `gradient_accumulation_steps`를 조정하세요.


In [2]:
import torch, platform

print(f"Python 버전: {platform.python_version()}")
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"총 VRAM: {total_vram:.1f} GB")
else:
    print("⚠️ GPU가 감지되지 않았습니다. NVIDIA 드라이버 / CUDA 설치를 확인하세요.")


Python 버전: 3.12.13
PyTorch 버전: 2.10.0+cu128
CUDA 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 4060 Laptop GPU
총 VRAM: 8.0 GB


## 1. 패키지 설치 및 버전 확인

처음 실행하는 환경이라면 필요한 패키지를 한 번만 설치합니다. 이미 설치되어 있다면 설치 셀은 건너뛰고, 바로 버전 확인 셀만 실행해도 됩니다.

이 노트북에서 사용하는 주요 패키지는 다음과 같습니다.

- `transformers`: Qwen 모델과 tokenizer 로드
- `peft`: LoRA 어댑터 구성 및 저장
- `trl`: `SFTTrainer`를 이용한 지도 미세조정
- `bitsandbytes`: `paged_adamw_8bit` optimizer 사용
- `datasets`: 학습 데이터셋 구성

> 여기서 `bitsandbytes`를 쓰지만, 이 노트북은 모델을 4bit로 양자화하지 않습니다. `bitsandbytes`는 optimizer 메모리 절약을 위해 사용됩니다.


In [3]:
# 최초 1회만 실행하세요. (앞의 # 을 지우고 실행)
# uv add "transformers>=5.10.1" "peft>=0.19.0" "trl>=0.24.0" "bitsandbytes>=0.48.0" \
#     "accelerate>=1.11.0" "datasets>=3.0.0" sentencepiece protobuf -U


In [4]:
import datasets

In [5]:
import transformers, peft, trl, bitsandbytes, accelerate

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [6]:
print("transformers :", transformers.__version__)
print("peft         :", peft.__version__)
print("trl          :", trl.__version__)
print("bitsandbytes :", bitsandbytes.__version__)
print("accelerate   :", accelerate.__version__)
print("datasets     :", datasets.__version__)


transformers : 5.5.0
peft         : 0.19.1
trl          : 0.24.0
bitsandbytes : 0.49.2
accelerate   : 1.14.0
datasets     : 4.3.0


## 2. 모델 로드 (양자화 없음 · bf16)

기본 모델은 **`Qwen/Qwen3-1.7B`** 입니다. 이 셀에서는 `BitsAndBytesConfig` 없이 모델을 그대로 `bf16`으로 GPU에 올립니다.

코드에서 중요한 부분은 다음과 같습니다.

- `torch_dtype=torch.bfloat16`: 모델을 bf16 정밀도로 로드합니다.
- `device_map={"": 0}`: 단일 GPU 0번에 모델을 명시적으로 올립니다.
- `tokenizer.pad_token`이 없으면 `eos_token`으로 대체합니다.

이번 실습의 목적은 양자화 기법이 아니라 **LoRA 어댑터가 답변 스타일을 어떻게 바꾸는지** 확인하는 것입니다.


In [7]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

# .env 파일 로드
load_dotenv(override=True)

HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)

print("모델 로드 완료! (양자화 없음, bf16)")
if torch.cuda.is_available():
    print(f"현재 GPU 메모리 사용량: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

모델 로드 완료! (양자화 없음, bf16)
현재 GPU 메모리 사용량: 3.20 GB


## 3. 응답 생성 함수

`ask()` 함수는 같은 질문을 파인튜닝 전과 후에 반복해서 테스트하기 위한 헬퍼 함수입니다.

- `tokenizer.apply_chat_template()`으로 Qwen 채팅 형식에 맞는 입력 문장을 만듭니다.
- `enable_thinking=False`로 thinking 모드가 아니라 일반 답변 모드로 생성합니다.
- `model.eval()`을 호출해 생성 중 LoRA dropout이 켜지지 않도록 합니다.
- `torch.no_grad()`로 추론 중 gradient 계산을 끕니다.


In [9]:
def ask(question, max_new_tokens=200):
    model.eval()  # LoRA dropout이 켜진 채 생성하면 답변이 불안정해지므로 항상 eval 모드로 고정
    messages = [{"role": "user", "content": question}] # 사용자 질문을 채팅 템플릿이 요구하는 메시지 형식으로 구성

    # 메시지를 모델이 학습한 채팅 프롬프트 형식의 문자열로 변환합니다.
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    # 프롬프트를 PyTorch 텐서로 토큰화한 뒤 모델이 위치한 장치로 이동합니다.
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # 추론 시에는 그래디언트 계산이 필요 없으므로 비활성화합니다.
    # 메모리 사용량이 줄고 생성 속도가 빨라집니다.
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens, # 새로 생성할 수 있는 최대 토큰 수
            do_sample=True, # 확률 분포에서 토큰을 샘플링하여 다양한 답변을 생성
            temperature=0.7, # 낮을수록 보수적이고 일관된 답변을 생성
            top_p=0.8, # 누적 확률이 top_p 이상인 토큰만 고려하여 샘플링
            repetition_penalty=1.15, # 동일한 토큰 반복을 억제
            no_repeat_ngram_size=3, # 동일한 3개 토큰 조합이 반복되지 않도록 제한
            pad_token_id=tokenizer.eos_token_id, # 별도의 PAD 토큰이 없을 경우 EOS 토큰을 패딩에 사용
        )

    # 생성된 토큰을 문자열로 변환합니다.
    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return response.strip()


## 4. Before: 파인튜닝 전 응답 확인

학습 전 모델이 같은 질문에 어떻게 답하는지 확인합니다. 이 결과는 `before_answers`에 저장해 두고, 학습 후 답변과 나란히 비교합니다.


In [10]:
test_questions = [
    "오늘 컨디션 어때?",
    "밥 먹었어?",
    "고마워요!",
    "요즘 스트레스가 심해",
    "주말에 뭐하지?"
]

before_answers = {}
print("=" * 60)
print("파인튜닝 전(Before) 응답")
print("=" * 60)
for q in test_questions:
    ans = ask(q)
    before_answers[q] = ans
    print(f"\nQ: {q}\nA: {ans}")
    print("-" * 50)


파인튜닝 전(Before) 응답

Q: 오늘 컨디션 어때?
A: 안녕하세요! 오늘은 여전히 건강한 상태로 잘 지내고 계시면 좋겠어요. 피로도가 있는 경우는 휴식을 취하고, 스트레스를 줄이기 위해 레저 활동이나 명상 등을 하세요. 만약 어떤 증상을 느끼거나 불편함이 있다면 언제든지 말씀해 주세요. 😊
--------------------------------------------------

Q: 밥 먹었어?
A: 밥을 먹었다고 했어. 어제는 아침에 밥을 머리 위로 놓았지만, 오늘은 집에서 일하고 있어서 밥이 없는데... 왜냐하면 내일부터는 밥도 안 먹을 거야. 😅
--------------------------------------------------

Q: 고마워요!
A: 안녕하세요! 고마워하시는 걸 보니 정말 마음에 드네요 😊  
질문이나 도움이 필요하시면 언제든지 물어보세요! 🌟
--------------------------------------------------

Q: 요즘 스트레스가 심해
A: 스트레스는 일상적인 삶에 영향을 줄 수 있지만, 지속적으로 높은 수준의 스트리스를 겪으면 건강에 해로울 수 있습니다. 아래는 스트RESS가 식별되고 관리를 위한 몇 가지 팁입니다:

---

### 🌿 1. **스트레즈 감지하기**
- **감정 상태 확인**: 어떤 감정이 강하게 느껴지는가? 불안, 우울, 분노 등.
- **신체 반응 확인**: 목소리, 허리 통증, 소화기 문제, 피로도 등을 주시하라.
- 측면에서 생각: "어떤 일이 발생했나?" 또는 "이제 어떻게 해결할 수 있을까?"

---

### 💡 2. **수행 가능한 작은 변화들**
- 매일 5분씩 명확한 목표를 설정하고 실행
--------------------------------------------------

Q: 주말에 뭐하지?
A: 주말에는 다양한 활동이 가능해요! 아래는 주말을 보내기 좋은 몇 가지 추천입니다:

### 1. **자연 속에서 즐기는 활동**
- 산책 또는 걷기: 

## 5. 데이터셋 준비

현대어 질문과 사극체 답변 쌍 40개를 학습 데이터로 사용합니다. 다음 셀에서는 원본 리스트를 Hugging Face `Dataset`으로 바꾼 뒤, 각 예제를 `prompt`와 `completion`으로 변환합니다.

- `prompt`: 사용자 질문과 assistant 시작 토큰까지 포함한 입력 부분
- `completion`: 모델이 학습해야 하는 사극체 답변 부분

뒤쪽 `SFTConfig`에서 `completion_only_loss=True`를 사용하므로, loss는 주로 `completion` 부분에만 계산됩니다. 즉, 모델은 질문 문장을 외우기보다 **assistant 답변 말투**를 배우게 됩니다.


In [11]:
sageuk_pairs = [
    [
        "안녕하세요!",
        "그대, 평안하셨는지요. 소인 인사 여쭙나이다."
    ],
    [
        "오늘 기분이 어때?",
        "소인의 기분을 물으시니 황공하옵니다. 오늘은 심신이 두루 평안하옵니다."
    ],
    [
        "잘 지냈어?",
        "예, 그간 별일 없이 잘 지냈사옵니다. 그대는 어찌 지내셨는지요?"
    ],
    [
        "반가워요",
        "소인 또한 그대를 뵙게 되어 참으로 반갑기 그지없사옵니다."
    ],
    [
        "처음 뵙겠습니다",
        "소인 이렇듯 그대를 처음 뵈오니 영광이옵니다."
    ],
    [
        "오늘 날씨 어때?",
        "오늘은 하늘이 청명하고 바람이 온화하여 나들이하기 참으로 좋은 날씨이옵니다."
    ],
    [
        "비 올까?",
        "하늘의 기색을 살피건대 오늘 중으로 비가 내릴 듯하니, 우산을 챙기심이 좋을 듯하옵니다."
    ],
    [
        "너무 더워",
        "그러하옵니다, 오늘따라 더위가 자못 심하니 몸을 각별히 살피셔야 하옵니다."
    ],
    [
        "겨울이 왔나봐",
        "찬바람이 부는 것을 보니 어느덧 겨울이 당도하였나 보옵니다."
    ],
    [
        "황사가 심해",
        "오늘은 흙먼지가 자욱하여 밖 나들이는 삼가시는 것이 좋을 듯하옵니다."
    ],
    [
        "점심 뭐 먹지?",
        "점심 수라로는 든든한 국밥 한 그릇이 어떠하올지요."
    ],
    [
        "맛집 추천해줘",
        "근방에 명성이 자자한 국밥집이 있사온데, 그곳을 한번 찾아보심이 어떠하올는지요."
    ],
    [
        "배고파",
        "시장하시다니 어서 요기를 하셔야 하옵니다. 곧 수라상을 봐 올리겠나이다."
    ],
    [
        "커피 마실래",
        "차 한 잔 올리오리까? 향긋한 다과와 함께 올리겠나이다."
    ],
    [
        "다이어트 중이야",
        "몸을 가벼이 하고자 절식을 행하고 계시는군요. 무리하지 마시고 건강을 먼저 살피소서."
    ],
    [
        "고마워",
        "별말씀을 다 하시옵니다. 소인이 마땅히 해야 할 도리를 다했을 뿐이옵니다."
    ],
    [
        "미안해",
        "괘념치 마시옵소서. 그럴 수도 있는 일이니 심려치 마소서."
    ],
    [
        "도와줘서 감사합니다",
        "도움이 되었다니 소인 또한 기쁘기 한량없사옵니다."
    ],
    [
        "실수했어",
        "누구든 실수는 있는 법이오니, 너무 자책하지 마시옵소서."
    ],
    [
        "늦어서 죄송해요",
        "늦으셨다 하나 크게 흠 될 일은 아니오니 편히 앉으시옵소서."
    ],
    [
        "이거 좀 도와줄래?",
        "여부가 있겠사옵니까. 소인이 힘닿는 데까지 도와드리겠나이다."
    ],
    [
        "설명해줘",
        "소인이 아는 바를 소상히 아뢰어 올리겠나이다."
    ],
    [
        "이해가 안 돼",
        "어려우시다면 다시 한번 쉽게 풀어 아뢰어 올리겠나이다."
    ],
    [
        "빨리 처리해줘",
        "급한 일이시라니 서둘러 처결하겠나이다."
    ],
    [
        "이거 확인해줄 수 있어?",
        "소인이 살펴보고 곧 아뢰어 올리겠나이다."
    ],
    [
        "내일 뭐해?",
        "내일은 특별한 일정이 없으니 그대의 뜻대로 움직이시면 될 듯하옵니다."
    ],
    [
        "약속 있어",
        "그러시다면 시각에 늦지 않도록 채비를 서두르심이 좋을 듯하옵니다."
    ],
    [
        "여행 가고 싶어",
        "먼 길 떠나고자 하시는 그 뜻, 참으로 좋은 생각이시옵니다."
    ],
    [
        "휴가 계획 세워줘",
        "며칠간의 여정을 소상히 짜 올리겠나이다."
    ],
    [
        "회의 몇 시야?",
        "회의는 오시(午時), 즉 정오 무렵으로 잡혀 있사옵니다."
    ],
    [
        "힘들어",
        "그간 고초가 많으셨나 보옵니다. 잠시 쉬어가심이 어떠하올는지요."
    ],
    [
        "슬퍼",
        "마음이 무거우시다니 소인의 마음 또한 편치 않사옵니다. 곁에서 위로하여 드리고 싶나이다."
    ],
    [
        "화가 나",
        "노여움이 크신 듯하오니 잠시 숨을 고르시고 마음을 가라앉히시옵소서."
    ],
    [
        "걱정돼",
        "심려가 크신가 보옵니다. 너무 근심치 마소서, 다 잘 풀릴 것이옵니다."
    ],
    [
        "행복해",
        "그 기쁨을 뵈오니 소인의 마음도 절로 흐뭇하옵니다."
    ],
    [
        "취미가 뭐야?",
        "소인은 옛 서책을 읽고 시문을 짓는 것을 낙으로 삼고 있사옵니다."
    ],
    [
        "영화 추천해줘",
        "근래 평판이 자자한 작품이 있사온데, 한번 감상해 보심이 어떠하올는지요."
    ],
    [
        "음악 들을래",
        "가야금 가락이라도 청해 올릴깝쇼."
    ],
    [
        "게임 좋아해?",
        "소인은 장기와 바둑을 즐겨 두는 편이옵니다."
    ],
    [
        "책 추천해줘",
        "마음의 양식이 될 만한 좋은 서책 한 권을 골라 올리겠나이다."
    ]
]

print(f"학습 데이터 개수: {len(sageuk_pairs)}개")
print("예시:", sageuk_pairs[0])


학습 데이터 개수: 40개
예시: ['안녕하세요!', '그대, 평안하셨는지요. 소인 인사 여쭙나이다.']


In [12]:
# Hugging Face의 데이터셋 생성 및 처리를 위한 Dataset 클래스를 불러옵니다.
from datasets import Dataset

# (질문, 답변) 형태의 sageuk_pairs를 instruction과 response 필드를 가진 딕셔너리 리스트로 변환합니다.
raw_data = [{"instruction": q, "response": a} for q, a in sageuk_pairs]

def format_example(example):
    """
    하나의 학습 데이터를 채팅 모델 학습에 적합한
    prompt-completion 형식으로 변환합니다.
    """

    # 사용자 질문을 모델 고유의 채팅 템플릿에 맞는 문자열로 변환합니다.
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": example["instruction"]}],
        tokenize=False, # 토큰 ID가 아닌 문자열 형태로 반환
        add_generation_prompt=True, # 모델의 답변 시작을 나타내는 프롬프트 추가
        enable_thinking=False, # 별도의 추론 과정(thinking) 출력을 비활성화
    )

    # 정답 마지막에 EOS(문장 종료) 토큰을 추가합니다.
    # 이를 통해 모델이 답변을 마쳐야 하는 시점을 학습할 수 있습니다.
    completion = example["response"] + tokenizer.eos_token

    # 프롬프트와 정답을 분리한 형태로 반환합니다.
    return {"prompt": prompt, "completion": completion}

# 딕셔너리 리스트를 Hugging Face Dataset 객체로 변환합니다.
dataset = Dataset.from_list(raw_data)

# 데이터셋의 각 예제에 format_example 함수를 적용합니다.
# 기존 instruction, response 열은 유지되고, 새로 생성한 prompt, completion 열이 추가됩니다
dataset = dataset.map(format_example)

# 변환 결과가 올바른지 첫 번째 데이터의 프롬프트를 확인합니다.
print("PROMPT:", dataset[0]["prompt"])
# 첫 번째 데이터의 정답과 EOS 토큰 추가 여부를 확인합니다.
print("COMPLETION:", dataset[0]["completion"])


Map:   0%|          | 0/40 [00:00<?, ? examples/s]

PROMPT: <|im_start|>user
안녕하세요!<|im_end|>
<|im_start|>assistant
<think>

</think>


COMPLETION: 그대, 평안하셨는지요. 소인 인사 여쭙나이다.<|im_end|>


## 6. LoRA 어댑터 설정 (양자화 없음)

이 노트북은 QLoRA가 아니므로 `prepare_model_for_kbit_training()`을 사용하지 않습니다. 대신 다음 순서로 순정 LoRA 학습을 준비합니다.

1. `model.gradient_checkpointing_enable()`로 메모리 사용량을 줄입니다.
2. `model.enable_input_require_grads()`로 gradient checkpointing 환경에서 입력 gradient를 허용합니다.
3. `LoraConfig`로 어댑터 설정을 정의합니다.
4. `get_peft_model()`로 원본 모델에 LoRA 어댑터를 붙입니다.

| 파라미터 | 의미 | 이번 실습 값 |
|---|---|---|
| `r` | LoRA rank. 클수록 표현력과 메모리 사용량이 증가합니다. | 16 |
| `lora_alpha` | LoRA scaling 계수입니다. | 32 |
| `lora_dropout` | 과적합 방지용 dropout입니다. | 0.05 |
| `target_modules` | LoRA를 붙일 attention/MLP projection layer입니다. | `q/k/v/o`, `gate/up/down` |

`model.print_trainable_parameters()` 결과에서 학습 가능한 파라미터가 0보다 커야 정상입니다. 0으로 나오면 LoRA 어댑터가 학습 대상이 아니므로, 모델 로드 셀부터 이 셀까지 순서대로 다시 실행하세요.


In [13]:
# PEFT(Parameter-Efficient Fine-Tuning) 라이브러리에서 LoRA 설정 클래스와 모델에 LoRA를 적용하는 함수를 불러옵니다.
from peft import LoraConfig, get_peft_model

# 순전파 중간 활성화 값을 모두 저장하지 않고, 역전파 시 필요한 값을 다시 계산하도록 설정합니다.
# 학습 속도가 다소 느려질 수 있지만 GPU 메모리 사용량을 줄일 수 있습니다.
model.gradient_checkpointing_enable()

# 양자화된 모델 등을 LoRA로 학습할 때 역전파가 가능하도록 입력 임베딩의 출력에 gradient 계산을 활성화합니다.
model.enable_input_require_grads()

# y = W x + (α / r) B A x
# W x              = 원본 모델의 출력
# (α / r) B A x    = LoRA adapter가 만든 보정 출력
# α / r은 LoRA 보정 출력이 원본 출력에 얼마나 강하게 반영될지 조절하는 값
# scale = (α / r)
# r : r = adapter 크기, 표현력, 병목 차원
# α : LoRA adapter의 영향력 크기를 조절하는 하이퍼파라미터
# α는 adapter의 학습 파라미터 수를 늘리는 값이 아니다. 오직 보정값의 세기를 조절한다.

# 모델에 적용할 LoRA 어댑터의 설정을 정의합니다.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,# LoRA 경로에 적용할 dropout 비율로 학습 중 과적합을 줄이는 데 도움을 줄 수 있으며, model.eval() 상태에서는 비활성화됩니다.
    bias="none", # 기존 선형 계층의 bias는 학습하지 않습니다. 따라서 주로 LoRA의 A, B 행렬만 학습됩니다
    task_type="CAUSAL_LM", # 다음 토큰을 예측하는 인과적 언어 모델임을 지정합니다.
    target_modules=[
        # LoRA 어댑터를 삽입할 선형 계층의 이름을 지정합니다.
        # Self-Attention 계층
        "q_proj", # Query 투영 계층
        "k_proj", # Key 투영 계층
        "v_proj", # Value 투영 계층
        "o_proj", # Attention 출력 투영 계층

        # MLP 또는 Feed-Forward 계층
        "gate_proj", # 게이트 투영 계층
        "up_proj", # 중간 차원을 확장하는 투영 계층
        "down_proj", # 확장된 차원을 다시 줄이는 투영 계층
    ],
)

# 기존 모델의 지정된 target_modules에 LoRA 어댑터를 삽입합니다.
# 반환된 모델은 원본 가중치 대부분이 고정되고,
# LoRA 어댑터 파라미터를 중심으로 학습되는 PEFT 모델입니다.
model = get_peft_model(model, lora_config)

# 전체 파라미터 수, 학습 가능한 파라미터 수와 비율을 출력하여 LoRA가 정상적으로 적용되었는지 확인합니다.
model.print_trainable_parameters()


trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


## 7. 학습 설정 및 실행

`SFTTrainer`로 사극체 답변을 지도 미세조정합니다. 이 실습은 작은 데이터셋으로 스타일 변화를 확인하는 것이 목적이므로, 전체 모델이 아니라 LoRA 어댑터만 업데이트합니다.

주요 설정은 다음과 같습니다.

- `per_device_train_batch_size=1`: GPU 메모리를 아끼기 위해 실제 배치는 1로 둡니다.
- `gradient_accumulation_steps=8`: 8번의 mini-batch를 모아 한 번 업데이트합니다.
- `learning_rate=2e-4`: LoRA 학습에서 자주 쓰는 범위의 학습률입니다.
- `bf16=True`: bf16 연산을 사용합니다.
- `optim="paged_adamw_8bit"`: optimizer 상태 메모리를 절약합니다.
- `max_length=512`: 입력과 출력 토큰을 합친 최대 길이입니다.
- `completion_only_loss=True`: 질문이 아니라 assistant 답변 부분 위주로 loss를 계산합니다.

학습이 끝나면 `model.eval()`로 전환합니다. LoRA dropout이 켜진 train mode에서 생성하면 답변이 불안정해질 수 있기 때문입니다.

> 오류 점검: loss가 전혀 줄지 않거나 학습이 되는 것처럼 보이는데 결과가 변하지 않으면, `model.print_trainable_parameters()`에서 trainable parameter가 0이 아닌지 먼저 확인하세요.


In [14]:
# TRL 라이브러리에서 지도 미세 조정(SFT)에 사용하는 Trainer 클래스와 학습 설정 클래스를 불러옵니다.
from trl import SFTTrainer, SFTConfig

# SFT(Supervised Fine-Tuning) 학습에 사용할 설정을 정의합니다.
training_args = SFTConfig(
    output_dir="./output/sample_a1_sageuk_lora", # 체크포인트와 학습 결과가 저장될 디렉터리
    num_train_epochs=8, # 전체 학습 데이터셋을 반복해서 학습할 횟수
    per_device_train_batch_size=1, # GPU 한 장이 한 번의 순전파·역전파에서 처리할 데이터 수
    gradient_accumulation_steps=8,  # 8개 미니 배치의 gradient를 누적한 뒤 한 번 가중치를 업데이트합니다. # GPU가 1개라면 실질적인 배치 크기는 1 × 8 = 8입니다.
    gradient_checkpointing=True, # 중간 활성화 값을 저장하는 대신 역전파 시 다시 계산하여 GPU 메모리 사용량을 줄입니다. 대신 학습 시간이 다소 증가합니다.
    learning_rate=2e-4, # 옵티마이저가 학습 파라미터를 한 번에 얼마나 변경할지 결정합니다. LoRA 학습에서 흔히 사용되는 비교적 높은 학습률입니다.
    logging_steps=5, # 5번의 학습 스텝마다 loss 등의 학습 상태를 출력합니다.
    save_strategy="no", # 학습 도중 체크포인트를 자동으로 저장하지 않습니다. 따라서 학습 후 어댑터가 필요하면 별도로 save_pretrained()를 호출해야 합니다.
    bf16=True, # bfloat16 정밀도를 사용하여 메모리 사용량을 줄이고 학습 속도를 높입니다. 사용 중인 GPU가 BF16 연산을 지원해야 합니다.
    optim="paged_adamw_8bit", # 8비트 paged AdamW 옵티마이저를 사용하여 일반 AdamW보다 옵티마이저 상태가 차지하는 GPU 메모리를 줄입니다.
    max_length=512, # 하나의 학습 예제가 가질 수 있는 최대 토큰 길이로, 이를 초과하는 입력은 잘릴 수 있습니다.
    completion_only_loss=True, # prompt 부분은 loss 계산에서 제외하고 completion, 즉 정답 부분에 대해서만 loss 계산에 포함합니다.
    report_to="none", # Weights & Biases 등의 외부 실험 추적 서비스에 로그를 전송하지 않습니다.
)
# 설정한 모델, 학습 옵션, 데이터셋을 사용하여 지도 미세 조정용 Trainer를 생성합니다.
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# 실제 SFT 학습을 시작합니다.
# completion_only_loss=True이므로 정답 부분을 중심으로 학습합니다.
trainer.train()

# 학습이 끝나면 반드시 eval 모드로 전환합니다.
# (LoRA dropout이 생성 중에도 계속 켜져 있으면 답변이 불안정해집니다)
model.eval()
print("학습 완료! model.eval() 적용됨 — 이제 안정적으로 답변을 생성할 수 있습니다.")


Adding EOS to train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

{'loss': '4.001', 'grad_norm': '0.4233', 'learning_rate': '0.00018', 'entropy': '1.754', 'num_tokens': '1783', 'mean_token_accuracy': '0.3396', 'epoch': '1'}
{'loss': '2.705', 'grad_norm': '0.3617', 'learning_rate': '0.000155', 'entropy': '2.889', 'num_tokens': '3566', 'mean_token_accuracy': '0.4332', 'epoch': '2'}
{'loss': '2.097', 'grad_norm': '0.2706', 'learning_rate': '0.00013', 'entropy': '2.656', 'num_tokens': '5349', 'mean_token_accuracy': '0.5386', 'epoch': '3'}
{'loss': '1.676', 'grad_norm': '0.2748', 'learning_rate': '0.000105', 'entropy': '2.298', 'num_tokens': '7132', 'mean_token_accuracy': '0.6214', 'epoch': '4'}
{'loss': '1.347', 'grad_norm': '0.3189', 'learning_rate': '8e-05', 'entropy': '2.08', 'num_tokens': '8915', 'mean_token_accuracy': '0.6903', 'epoch': '5'}
{'loss': '1.124', 'grad_norm': '0.2282', 'learning_rate': '5.5e-05', 'entropy': '1.909', 'num_tokens': '1.07e+04', 'mean_token_accuracy': '0.745', 'epoch': '6'}
{'loss': '0.9398', 'grad_norm': '0.2652', 'learnin

## 8. After: 파인튜닝 후 응답 비교

학습 전에 저장해 둔 `before_answers`와 학습 후 `ask()` 결과를 나란히 출력합니다. 사극체 어미와 표현이 일관되게 나타나는지 확인합니다.


In [15]:
print("=" * 60)
print("Before vs After 비교")
print("=" * 60)
for q in test_questions:
    after_ans = ask(q)
    print(f"\nQ: {q}")
    print(f"[Before] {before_answers[q]}")
    print(f"[After ] {after_ans}")
    print("-" * 50)


Before vs After 비교

Q: 오늘 컨디션 어때?
[Before] 안녕하세요! 오늘은 여전히 건강한 상태로 잘 지내고 계시면 좋겠어요. 피로도가 있는 경우는 휴식을 취하고, 스트레스를 줄이기 위해 레저 활동이나 명상 등을 하세요. 만약 어떤 증상을 느끼거나 불편함이 있다면 언제든지 말씀해 주세요. 😊
[After ] 마음이 상한 상태가 아니시면 편히 앉아 올리심을 바라옵니다.
--------------------------------------------------

Q: 밥 먹었어?
[Before] 밥을 먹었다고 했어. 어제는 아침에 밥을 머리 위로 놓았지만, 오늘은 집에서 일하고 있어서 밥이 없는데... 왜냐하면 내일부터는 밥도 안 먹을 거야. 😅
[After ] 아니오, 오늘은 절식으로 그릇을 채워 올리겠나이다.
--------------------------------------------------

Q: 고마워요!
[Before] 안녕하세요! 고마워하시는 걸 보니 정말 마음에 드네요 😊  
질문이나 도움이 필요하시면 언제든지 물어보세요! 🌟
[After ] 소인도 그대의 흠미가 있으니 나를 부담스럽게 하셔서 마음이 둘러珑하옵니다. 소인이 실로 미안하올 참으로 그러시옵니다만, 곧 화해할 것이옵니다.
(그러나 어려움을 겪으셨는지 몰라하여 속상치 마찬가위여서 옆에서 간사히 도와드릴 수도 있겠나이다.)
--------------------------------------------------

Q: 요즘 스트레스가 심해
[Before] 스트레스는 일상적인 삶에 영향을 줄 수 있지만, 지속적으로 높은 수준의 스트리스를 겪으면 건강에 해로울 수 있습니다. 아래는 스트RESS가 식별되고 관리를 위한 몇 가지 팁입니다:

---

### 🌿 1. **스트레즈 감지하기**
- **감정 상태 확인**: 어떤 감정이 강하게 느껴지는가? 불안, 우울, 분노 등.
- **신체 반응 확인**: 목소리, 허리 통증, 소화기 문제, 피로도 등을 주시하라.
- 측면에서

## 9. LoRA 어댑터 저장

이 셀은 전체 모델이 아니라 **LoRA 어댑터만** 저장합니다. 저장 위치는 다음과 같습니다.

```text
./lora_adapters/exam4_sageuk_lora
```

어댑터만 저장하면 파일 크기가 작고, 나중에 같은 base model에 다시 붙여서 사용할 수 있습니다. 전체 모델로 배포하려면 별도의 병합 과정이 필요합니다.


In [16]:
ADAPTER_DIR = "./lora_adapters/exam4_sageuk_lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA 어댑터 저장 완료: {ADAPTER_DIR}")


LoRA 어댑터 저장 완료: ./lora_adapters/exam4_sageuk_lora


## 정리

이 노트북에서 확인한 내용은 다음과 같습니다.

- 양자화 없이 `Qwen/Qwen3-1.7B`를 bf16으로 로드했습니다.
- 원본 모델 전체를 학습하지 않고 LoRA 어댑터만 학습했습니다.
- `prompt` / `completion` 데이터 구조와 `completion_only_loss=True`를 사용해 assistant 답변 말투를 중심으로 학습했습니다.
- 파인튜닝 전/후 답변을 비교해 스타일 변화가 실제로 일어나는지 확인했습니다.
- 최종 산출물로 LoRA 어댑터를 저장했습니다.

다음 단계로는 저장된 어댑터를 다시 불러오거나, base model과 병합한 뒤 Ollama 같은 로컬 실행 환경에 배포할 수 있습니다.
